### Import Dependencies

In [1]:
import uuid

import httpx
from a2a.client import A2ACardResolver, create_client
from a2a.types import (
    AgentCard,
    Message,
    Part,
    Role,
    SendMessageRequest
)
from google.protobuf.json_format import MessageToJson

### Test the connection

In [2]:
BASE_URL = "http://localhost:10001"
PUBLIC_AGENT_CARD_PATH = "/.well-known/agent-card.json"

In [4]:
async with httpx.AsyncClient() as httpx_client:

    resolver = A2ACardResolver(
        httpx_client=httpx_client,
        base_url=BASE_URL
    )

    try:
        print(f"Fetching public agent card from: {BASE_URL}{PUBLIC_AGENT_CARD_PATH}")
        public_card = await resolver.get_agent_card()
        print("Fetched public agent card")
        print(MessageToJson(public_card, indent=2))

    except Exception as e:
        print(f"Error fetching public agent card: {e}")

Fetching public agent card from: http://localhost:10001/.well-known/agent-card.json
Fetched public agent card
{
  "name": "warehouse_manager_agent",
  "description": "The user is asking to reserve items from the warehouses or about availability of the items in warehouses.",
  "supportedInterfaces": [
    {
      "url": "http://localhost:10001/",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "1.0.0",
  "capabilities": {
    "streaming": true
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "skills": [
    {
      "id": "ABC",
      "name": "Check Availability",
      "description": "Check availability of items across warehouses.",
      "tags": [
        "availability",
        "warehouse"
      ],
      "examples": [
        "What is the availability of the item 123?"
      ]
    },
    {
      "id": "DEF",
      "name": "Reserve Items",
      "description": "Reserve items from multiple warehouses in 

### Run commands against the Agent

In [6]:
timeout_config = httpx.Timeout(
    connect=20.0,   # Connection timeout
    read=100.0,     # Read timeout (important for long-running operations)
    write=10.0,     # Write timeout
    pool=10.0       # Pool timeout
)

client = await create_client(
    agent=BASE_URL,
    resolver_http_kwargs={"timeout": timeout_config}
)

print("A2AClient initialized")

message_id = str(uuid.uuid4())
message_payload = Message(
    role=Role.ROLE_USER,
    message_id=message_id,
    parts=[Part(text="Hello, how are you?")],
)
request = SendMessageRequest(message=message_payload)

async for response in client.send_message(request):
    print(response)

A2AClient initialized
task {
  id: "50a14cfb-4dec-4c83-88fd-3662a231767f"
  context_id: "e01fb4b6-de77-41b2-b3cb-09251ecc3e30"
  status {
    state: TASK_STATE_SUBMITTED
  }
}

status_update {
  task_id: "50a14cfb-4dec-4c83-88fd-3662a231767f"
  context_id: "e01fb4b6-de77-41b2-b3cb-09251ecc3e30"
  status {
    state: TASK_STATE_WORKING
    timestamp {
      seconds: 1789634327
      nanos: 646147000
    }
  }
}

artifact_update {
  task_id: "50a14cfb-4dec-4c83-88fd-3662a231767f"
  context_id: "e01fb4b6-de77-41b2-b3cb-09251ecc3e30"
  artifact {
    artifact_id: "ee3fc35c-dbbf-4ba2-87a7-9f3c9e9bb8a3"
    parts {
      text: "Hello! I\'m doing well—thanks for asking. How can I help you today?"
    }
  }
}

status_update {
  task_id: "50a14cfb-4dec-4c83-88fd-3662a231767f"
  context_id: "e01fb4b6-de77-41b2-b3cb-09251ecc3e30"
  status {
    state: TASK_STATE_COMPLETED
    timestamp {
      seconds: 1789634329
      nanos: 34936000
    }
  }
}



In [8]:
async def run_a2a_warehouse_agent(query: str):

    timeout_config = httpx.Timeout(
        connect=20.0,   # Connection timeout
        read=100.0,     # Read timeout (important for long-running operations)
        write=10.0,     # Write timeout
        pool=10.0       # Pool timeout
    )

    client = await create_client(
        agent=BASE_URL,
        resolver_http_kwargs={"timeout": timeout_config}
    )


    message_id = str(uuid.uuid4())
    message_payload = Message(
        role=Role.ROLE_USER,
        message_id=message_id,
        parts=[Part(text=query)],
    )
    request = SendMessageRequest(message=message_payload)

    final_text_parts = []

    async for response in client.send_message(request):
        if response.HasField("artifact_update"):
            for part in response.artifact_update.artifact.parts:
                if part.text:
                    final_text_parts.append(part.text)

    final_response = "".join(final_text_parts)

    return final_response


In [9]:
answer_1 = await run_a2a_warehouse_agent("What is the availability of B09X1LDMH6 in all of your warehouses?")

In [10]:
print(answer_1)

Actions performed:
1. Checked availability of product **B09X1LDMH6** across all warehouses.
2. Result: **No stock available in any warehouse**.

Availability summary:
- **Total available across warehouses:** 0
- **Warehouses with full fulfillment:** None
- **Warehouses with partial fulfillment:** None

Order status:
- **This item cannot be reserved** because it is unavailable in all warehouses.


In [11]:
answer_1 = await run_a2a_warehouse_agent("can you reserve 1 of B09X1LDMH6 in Hamburg?")

In [12]:
print(answer_1)

Action performed: checked warehouse availability for 1× B09X1LDMH6 in Hamburg.

Result: the item cannot be reserved.
- Total available across warehouses: 0
- Hamburg warehouse: 0 available
- No warehouse has stock for this item

Reservation status: not possible.
